# Quality Baseline Worker Implementation & Rollup Evaluation
This notebook verifies the mathematical and database adapter integrations for **F-Q-09** (Rolling Baseline Recomputation) and **F-Q-10** (Quality Trend Rollup) under the LLM Observability Platform.

In [ ]:
import sys
import os
from datetime import datetime, date, timezone, timedelta
import pandas as pd
import matplotlib.pyplot as plt

# Add worker package src directory to python path
sys.path.append(os.path.abspath('../../../../packages/python/quality-baseline-worker/src'))

from features.quality_baseline.service import QualityBaselineService
from features.quality_baseline.types import BaselineRecomputeResult, DailyRollupRecord
print("Service and types loaded successfully.")

## 1. Mocking Ports & Adapters
We implement lightweight mocks for our Hexagonal Ports (`PostgresPort`, `RedisPort`, `ClickHousePort`, `MetricsPort`) to run out-of-band simulations.

In [ ]:
class MockPostgres:
    def __init__(self, baselines, rollup_data):
        self.baselines = baselines
        self.rollup_data = rollup_data

    def get_rolling_baselines(self):
        return self.baselines

    def get_daily_rollup_data(self, start_time, end_time):
        return self.rollup_data

class MockRedis:
    def __init__(self):
        self.calls = []

    def set_baseline_quality(self, model, endpoint, prompt_type, score):
        self.calls.append({
            "model": model,
            "endpoint": endpoint,
            "prompt_type": prompt_type,
            "score": score
        })

class MockClickHouse:
    def __init__(self):
        self.calls = []

    def insert_quality_trends(self, rows):
        self.calls.extend(rows)

class MockMetrics:
    def __init__(self):
        self.latency_calls = []
        self.updates_calls = []
        self.rollup_calls = []

    def record_workflow_run_latency(self, workflow_name, latency_ms):
        self.latency_calls.append((workflow_name, latency_ms))

    def record_baseline_updates(self, count):
        self.updates_calls.append(count)

    def record_rollup_rows(self, count):
        self.rollup_calls.append(count)

## 2. Simulating Baseline Recomputation (F-Q-09)
We simulate the calculation of 7-day average baseline quality scores for multiple models. 
We model two scenarios:
1. **Stable Model** (`GPT-4`): Average quality remains steady at `0.92` with `15,000` samples.
2. **Drifting Model** (`Llama-3-FineTuned`): Average quality drifts downward to `0.65` under a series of silent degradations.

In [ ]:
baselines_data = [
    BaselineRecomputeResult(model="gpt-4", endpoint="v1/chat", prompt_type="rag-retrieval", avg_score=0.92, sample_count=15000),
    BaselineRecomputeResult(model="llama-3-ft", endpoint="v1/completions", prompt_type="reasoning", avg_score=0.65, sample_count=8500)
]

pg = MockPostgres(baselines_data, [])
redis = MockRedis()
metrics = MockMetrics()

count = QualityBaselineService.recompute_and_update_redis(pg, redis, metrics)
print(f"Recomputed baselines written to Redis: {count}")
print("Redis Cache writes:")
print(pd.DataFrame(redis.calls))

## 3. Daily Rollup Aggregation (F-Q-10)
We verify that yesterday's statistics are compiled and appended into ClickHouse for historical trending.

In [ ]:
yesterday = date.today() - timedelta(days=1)
rollup_data = [
    DailyRollupRecord(
        rollup_date=yesterday,
        model="gpt-4",
        endpoint="v1/chat",
        prompt_type="rag-retrieval",
        avg_composite_score=0.915,
        flag_count=12,
        sample_count=22000
    ),
    DailyRollupRecord(
        rollup_date=yesterday,
        model="llama-3-ft",
        endpoint="v1/completions",
        prompt_type="reasoning",
        avg_composite_score=0.641,
        flag_count=182,
        sample_count=12000
    )
]

pg_rollup = MockPostgres([], rollup_data)
ch = MockClickHouse()
metrics_rollup = MockMetrics()

count_rollup = QualityBaselineService.rollup_yesterday_scores(pg_rollup, ch, metrics_rollup, target_date=datetime.now(timezone.utc))
print(f"Rollup records written to ClickHouse: {count_rollup}")
print("ClickHouse records:")
for call in ch.calls:
    print(call)

## 4. Visualizing Drift & Alert Thresholds
We plot simulated daily quality averages for a 30-day period. 
A baseline is calculated dynamically. If the daily average falls below the baseline by more than 15%, we trigger a mock alert.

In [ ]:
import numpy as np

# Generate 30 days of data
np.random.seed(42)
days = np.arange(1, 31)
dates = [date.today() - timedelta(days=int(30 - d)) for d in days]

# Scenario: Model degrades on day 20 due to an updated retrieval database
scores_gpt4 = np.concatenate([
    np.random.normal(0.92, 0.01, 19),
    np.random.normal(0.72, 0.03, 11)  # Drop
])

df = pd.DataFrame({
    "date": dates,
    "score": scores_gpt4
})

# Calculate rolling baseline (7-day window trailing)
df["baseline"] = df["score"].rolling(window=7, min_periods=1).mean()

# Alert check
df["alert_triggered"] = df["score"] < (df["baseline"] * 0.85)

# Plotting
plt.figure(figsize=(10, 5))
plt.plot(df["date"], df["score"], label="Daily Avg Quality Score", marker='o', color='royalblue')
plt.plot(df["date"], df["baseline"], label="7-Day Rolling Baseline", linestyle='--', color='darkorange')

# Highlight alerts
alerts = df[df["alert_triggered"]]
plt.scatter(alerts["date"], alerts["score"], color='red', label="Alert: >15% Drop Below Baseline", zorder=5, s=100, edgecolors='black')

plt.title("F-Q-09/10: Model Quality Baseline Drift & Degradation Alerting")
plt.xlabel("Date")
plt.ylabel("Composite Quality Score")
plt.axhline(0.5, color='gray', linestyle=':', label="Toxicity Scorer Threshold (0.50)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()